# ViTASD Baseline Reproduction (Colab Free)

Kiểm tra baseline `train_vitasd_baseline.py` (PhoBERT + multi-head attention,
best-effort reproduction của ViTASD) trước khi dùng làm điểm so sánh cho 2
module đóng góp (Text Normalization + Imbalanced Learning).

**Mục tiêu:** ra macro F1 gần 61.77% (mobile) / 41.12% (restaurant) / 52.64%
(hotel). Nếu gần → baseline hợp lý, đi tiếp bước ablation đầy đủ
(`colab_pair_ablation.ipynb`). Nếu vẫn cách xa → cần source code/paper gốc từ
thầy Kiệt, không đoán thêm được nữa. Xem docstring đầu file
`train_vitasd_baseline.py` để biết phần nào đã xác nhận từ paper, phần nào là
giả định.

⏱️ Ước tính: smoke test ~5 phút. 1 domain × 10 epoch (PhoBERT, fp16, batch 32)
~20-40 phút tuỳ độ dài dataset domain. Cả 3 domain ~1.5-2 tiếng.

In [ ]:
# 1. Clone dataset từ ViTASA repo gốc
!git clone https://github.com/kh4nh12/ViTASA.git ViTASA_repo 2>&1 | grep -E '(Cloning|clone|done)'
!echo "Dataset files:"
!ls -lh ViTASA_repo/*.jsonl

In [ ]:
# 2. Install dependencies
!pip install -q torch transformers scikit-learn seqeval underthesea
import torch
print(f"✅ PyTorch {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# 3. Setup folder structure + copy dataset
import os
import shutil

os.makedirs('VITASA_Enhanced/baseline/data', exist_ok=True)
os.makedirs('VITASA_Enhanced/experiments/results_vitasd_repro', exist_ok=True)

for domain in ['mobile', 'restaurant', 'hotel']:
    domain_dir = f'VITASA_Enhanced/baseline/data/{domain}'
    os.makedirs(domain_dir, exist_ok=True)
    src = f'ViTASA_repo/{domain}.jsonl'
    dst = f'{domain_dir}/{domain}.jsonl'
    if os.path.exists(src):
        shutil.copy(src, dst)
        lines = sum(1 for _ in open(dst))
        print(f"✅ {domain}: {lines} samples")

In [ ]:
# 4. Mount Google Drive (project code nằm trong Drive)
from google.colab import drive
drive.mount('/gdrive')
print("✅ Google Drive mounted")

In [ ]:
# 5. Copy code files từ Drive
import shutil

# Giả sử bạn đã upload project vào Drive folder "VITASA_Enhanced"
src_drive = '/gdrive/My Drive/VITASA_Enhanced'

for fname in ['train_pair.py', 'train_vitasd_baseline.py']:
    ok = os.system(f"cp '{src_drive}/{fname}' VITASA_Enhanced/ 2>/dev/null") == 0
    print(f"{'✅' if ok else '⚠️ '} {fname} {'copied' if ok else 'NOT FOUND in Drive'}")

for folder in ['text_normalization', 'imbalanced_learning']:
    ok = os.system(f"cp -r '{src_drive}/{folder}' VITASA_Enhanced/ 2>/dev/null") == 0
    print(f"{'✅' if ok else '⚠️ '} {folder} {'copied' if ok else 'NOT FOUND in Drive'}")

!echo "Files in VITASA_Enhanced:" && ls -la VITASA_Enhanced/ | grep -E '(train_pair|train_vitasd|text_norm|imbalanced)'

In [ ]:
# 6. Smoke test — 1 epoch, 10% data, domain mobile
%cd VITASA_Enhanced

!python3 train_vitasd_baseline.py --domain mobile --epochs 1 --subsample 0.1 2>&1 | tail -40

print("\n✅ Nếu chạy tới đây không lỗi → pipeline OK, chạy full ở cell tiếp theo")

In [ ]:
# 7. Chạy baseline thật — từng domain, 10 epoch, fp16
# Chạy tuần tự mobile -> restaurant -> hotel (nhẹ -> nặng, bắt lỗi sớm)
import subprocess

DOMAINS = ['mobile', 'restaurant', 'hotel']
EPOCHS = 10

for domain in DOMAINS:
    print(f"\n{'='*70}")
    print(f"[{domain}] ViTASD baseline reproduction — {EPOCHS} epochs")
    print(f"{'='*70}")
    cmd = f"python3 train_vitasd_baseline.py --domain {domain} --epochs {EPOCHS} --fp16"
    result = subprocess.run(cmd.split(), capture_output=False)
    if result.returncode != 0:
        print(f"❌ ERROR: {domain} failed — dừng lại, kiểm tra log phía trên trước khi chạy tiếp")
        break

print("\n✅ Done!")

In [ ]:
# 8. So sánh với baseline paper
import json
from pathlib import Path

BASELINE = {"mobile": 61.77, "restaurant": 41.12, "hotel": 52.64}
results_dir = Path("experiments/results_vitasd_repro")

print(f"{'Domain':<12} {'F1 (loại none)':>16} {'F1 (có none)':>16} {'Baseline paper':>16} {'Chênh (loại none)':>20}")
print("-" * 85)
for domain in ['mobile', 'restaurant', 'hotel']:
    f = results_dir / f"vitasd_repro_{domain}_phobert" / "results.json"
    if not f.exists():
        print(f"{domain:<12} — chưa chạy —")
        continue
    d = json.load(open(f))
    f1 = d["test"]["macro_f1"] * 100
    f1n = d["test"]["macro_f1_with_none"] * 100
    baseline = BASELINE[domain]
    print(f"{domain:<12} {f1:>15.2f}% {f1n:>15.2f}% {baseline:>15.2f}% {f1-baseline:>+19.2f}")

print("\nNếu chênh lệch (loại none) gần 0 → baseline khớp paper, dùng ngay.")
print("Nếu vẫn cách xa nhiều (>10 điểm) → cần source code/paper gốc từ thầy Kiệt.")

In [ ]:
# 9. Download kết quả về máy
from google.colab import files

!tar -czf vitasd_baseline_results.tar.gz experiments/results_vitasd_repro/
!ls -lh vitasd_baseline_results.tar.gz

print("\n📥 Downloading...")
files.download("vitasd_baseline_results.tar.gz")
print("✅ Done!")

## Ghi chú

**Nếu số liệu vẫn cách xa baseline paper sau khi đã chạy đủ epoch:**
Đây là bản best-effort reproduction (xem docstring đầu `train_vitasd_baseline.py`)
— nhiều chi tiết (kiến trúc multi-head attention chính xác, cách xây auxiliary
sentence, macro F1 có tính lớp `none` không) đang là giả định vì paper gốc
chỉ đọc được phần preview. Việc tiếp theo là xin PDF đầy đủ hoặc source code
từ thầy Kiệt, không nên tiếp tục đoán/tune tay.

**Nếu Colab timeout giữa chừng:**
Chạy lại đúng domain bị dừng (data vẫn còn trong session nếu chưa disconnect
hẳn, hoặc chạy lại từ cell 3 nếu phải restart runtime).